# NQ holdout pull — 2023/2024 out-of-regime canon validation

Pulls MBP-10 depth + trades for the **pre-registered** holdout days and **condenses each day
in the same loop**, so only the small committable artifacts ever touch disk. The raw frames are
freed immediately and never written — that is the fix versus saving raw parquet and condensing
in a second pass.

Output per day is roughly **150 KB** instead of ~1 GB, so the final zip is small enough to hand
straight back to Claude.

| | |
|---|---|
| sealed days | **128** (63 in 2023, 65 in 2024) |
| blocks | 2023-07, 2023-09, 2023-11, 2024-03, 2024-04, 2024-10 |
| SHA-256 of day list | `f4e17f1770a4d5314d02ccdda7d362b9…` |

Fitted span of the canon is 2025-06-02 → 2026-07-15. Nothing below overlaps it.

**Three artifacts, three different consumers — the formats are not interchangeable:**

| output | window (native tz) | format | read by |
|---|---|---|---|
| `nq_depth_<day>_ny.csv` | 08:00–11:00 **America/New_York** | long `ts,side,price,size` | `scripts/trade_matrix.py`, `depth_features.py` |
| `glbx-mdp3-<yyyymmdd>.mbp-10_condensed.csv` | 08:00–10:00 **Europe/London** | wide, raw Databento columns | `scripts/london_depth.py` |
| `footprint_holdout_<yyyy-mm>.parquet` | 18:00 (D−1) → 11:00 (D) **ET** | `ts_minute,price,side,volume,trades` | `scripts/champion_journal_cvd.py` |

Windows are defined in their **native timezone** and localized per-day, so US and UK DST
transitions land correctly — including the March/November weeks where the two calendars disagree.

## 1 — Install and authenticate

Your key is entered at runtime and never stored in the notebook.

In [ ]:
!pip install -q databento pandas pyarrow

import os, time, zipfile, getpass, gc
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import databento as db
import pandas as pd
import numpy as np

API_KEY = getpass.getpass("Databento API key: ").strip()
client = db.Historical(API_KEY)
print("authenticated")

## 2 — Configuration

Three toggles for which artifacts to build, and `LIMIT_DAYS` for a smoke test. Edit anything
here and **re-run this cell** before continuing — editing alone changes nothing.

In [ ]:
DATASET = "GLBX.MDP3"
SYMBOL  = "NQ.v.0"          # front month, rolled by volume
STYPE   = "continuous"

# Which artifacts to build. Depth is the expensive one; trades is roughly 5-10% the size.
PULL_NY_DEPTH     = True    # canon pre + golden books
PULL_LONDON_DEPTH = True    # london canon book
PULL_TRADES       = True    # CVD / footprint - feeds checks F, C, Tc, BIGFD, LONSLOPE

# Smoke-test knob: set to 3 to pull only the first 3 days and check the plumbing,
# then set back to 0 and re-run - already-downloaded days are skipped, not re-fetched.
LIMIT_DAYS = 0              # 0 = all 128 sealed days

OUTDIR = "/content/holdout_out"
ZIPOUT = "/content/nq_holdout_2023_24_condensed.zip"

# Optional: survive a Colab disconnect by writing to Drive instead.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/holdout_out"
    ZIPOUT = "/content/drive/MyDrive/nq_holdout_2023_24_condensed.zip"

NY  = ZoneInfo("America/New_York")
LON = ZoneInfo("Europe/London")
UTC = ZoneInfo("UTC")

# (name, tz, start, end) - localized per day, so DST is handled by the calendar not by hand.
# NY ends 10:30, not 11:00 (ANGUS 2026-07-27): golden window is 09:40-10:15 and depth is only
# ever read AT the fill minute. Verified against output/canon_book.parquet - pre fills run
# 08:01-09:23, gold 09:41-10:12, and ZERO fills land at or after 10:15 anywhere in the
# universe including rejected candidates. 10:30 leaves 18 min of headroom and cuts ~17% off
# the MBP-10 bill. Trades below still run to 11:00 - the in-trade layer reads forward flow
# (fw_3) up to 10 minutes AFTER the fill, so the tape must outlast the book.
DEPTH_WINDOWS = {
    "ny":     (NY,  (8, 0), (10, 30)),   # pre-market + golden (09:40-10:15) + headroom
    "london": (LON, (8, 0), (10, 0)),    # first 2h London
}

for sub in ("depth_ny", "depth_london", "cvd"):
    os.makedirs(f"{OUTDIR}/{sub}", exist_ok=True)

# ---- the sealed, pre-registered day list -------------------------------------
# Generated by scripts/sample_holdout_days.py. Do not edit by hand: the SHA-256 in
# docs/HOLDOUT-2023-24-PREREGISTRATION.md is the commitment that these are the days.
DAYS = [
    "2023-07-03", "2023-07-04", "2023-07-05", "2023-07-06", "2023-07-07", "2023-07-10",
    "2023-07-11", "2023-07-12", "2023-07-13", "2023-07-14", "2023-07-17", "2023-07-18",
    "2023-07-19", "2023-07-20", "2023-07-21", "2023-07-24", "2023-07-25", "2023-07-26",
    "2023-07-27", "2023-07-28", "2023-07-31", "2023-09-01", "2023-09-04", "2023-09-05",
    "2023-09-06", "2023-09-07", "2023-09-08", "2023-09-11", "2023-09-12", "2023-09-13",
    "2023-09-14", "2023-09-15", "2023-09-18", "2023-09-19", "2023-09-20", "2023-09-21",
    "2023-09-22", "2023-09-25", "2023-09-26", "2023-09-27", "2023-09-28", "2023-09-29",
    "2023-11-01", "2023-11-02", "2023-11-03", "2023-11-06", "2023-11-07", "2023-11-08",
    "2023-11-09", "2023-11-10", "2023-11-13", "2023-11-14", "2023-11-15", "2023-11-16",
    "2023-11-17", "2023-11-20", "2023-11-21", "2023-11-22", "2023-11-23", "2023-11-27",
    "2023-11-28", "2023-11-29", "2023-11-30", "2024-03-01", "2024-03-04", "2024-03-05",
    "2024-03-06", "2024-03-07", "2024-03-08", "2024-03-11", "2024-03-12", "2024-03-13",
    "2024-03-14", "2024-03-15", "2024-03-18", "2024-03-19", "2024-03-20", "2024-03-21",
    "2024-03-22", "2024-03-25", "2024-03-26", "2024-03-27", "2024-03-28", "2024-04-01",
    "2024-04-02", "2024-04-03", "2024-04-04", "2024-04-05", "2024-04-08", "2024-04-09",
    "2024-04-10", "2024-04-11", "2024-04-12", "2024-04-15", "2024-04-16", "2024-04-17",
    "2024-04-18", "2024-04-19", "2024-04-22", "2024-04-23", "2024-04-24", "2024-04-25",
    "2024-04-26", "2024-04-29", "2024-04-30", "2024-10-01", "2024-10-02", "2024-10-03",
    "2024-10-04", "2024-10-07", "2024-10-08", "2024-10-09", "2024-10-10", "2024-10-11",
    "2024-10-14", "2024-10-15", "2024-10-16", "2024-10-17", "2024-10-18", "2024-10-21",
    "2024-10-22", "2024-10-23", "2024-10-24", "2024-10-25", "2024-10-28", "2024-10-29",
    "2024-10-30", "2024-10-31",
]
SEALED_SHA256 = "f4e17f1770a4d5314d02ccdda7d362b97ae891aa833ea21ef564fc5ca7c9a87e"

import hashlib
_check = hashlib.sha256("\n".join(DAYS).encode()).hexdigest()
assert _check == SEALED_SHA256, f"day list altered! {_check} != {SEALED_SHA256}"
print(f"{len(DAYS)} sealed days verified against pre-registration")
print(f"  {DAYS[0]} .. {DAYS[-1]}")

## 3 — Window helpers

Every request is built by localizing the wall-clock window into its own timezone and then
converting to UTC. `fold=0` pins the ambiguous hour on autumn fall-back days.

In [ ]:
def utc_window(day: str, tz, start_hm, end_hm):
    """Localize a wall-clock window in `tz` for `day` and return UTC ISO strings."""
    y, m, dd = (int(x) for x in day.split("-"))
    s = datetime(y, m, dd, *start_hm, tzinfo=tz)
    e = datetime(y, m, dd, *end_hm, tzinfo=tz)
    return (s.astimezone(UTC).strftime("%Y-%m-%dT%H:%M:%S"),
            e.astimezone(UTC).strftime("%Y-%m-%dT%H:%M:%S"))


def trades_window(day: str):
    """18:00 ET the PREVIOUS calendar day -> 11:00 ET on `day`.

    CVD features reach back through the overnight: cvd_ON, cvd_ASIA (18:00-03:00 ET),
    cvd_LON (03:00-09:30) and cvd_PM all sit inside this span. Consecutive sampled days
    never overlap (each window ends 11:00 and the next starts 18:00), so concatenating
    the per-day frames cannot double-count a minute.
    """
    y, m, dd = (int(x) for x in day.split("-"))
    end = datetime(y, m, dd, 11, 0, tzinfo=NY)
    start = datetime(y, m, dd, 18, 0, tzinfo=NY) - timedelta(days=1)
    return (start.astimezone(UTC).strftime("%Y-%m-%dT%H:%M:%S"),
            end.astimezone(UTC).strftime("%Y-%m-%dT%H:%M:%S"))


def _decode_px(s):
    """Databento prices are 1e-9 fixed-point ints; to_df() usually decodes them already."""
    s = pd.to_numeric(s, errors="coerce")
    return np.where(s.abs() > 1e7, s / 1e9, s)


def _front_only(df):
    """Keep the modal instrument_id - a continuous pull can straddle a roll."""
    if "instrument_id" in df.columns and df["instrument_id"].nunique() > 1:
        return df[df["instrument_id"] == df["instrument_id"].value_counts().idxmax()]
    return df


print(f"sanity - 2023-03-01 ny  window (UTC): {utc_window('2023-03-01', NY,  (8,0), (11,0))}")
print(f"sanity - 2023-03-01 lon window (UTC): {utc_window('2023-03-01', LON, (8,0), (10,0))}")
print(f"sanity - 2023-03-01 trades   (UTC): {trades_window('2023-03-01')}")
print("note the March gap: US is already on DST, UK is not - hence the differing offsets")

## 4 — The three condensers

Definitions only, nothing runs. Each turns one raw Databento frame into the exact shape its
consumer in the repo expects — the formats are pinned by `tests/test_colab_holdout_notebook.py`.

In [ ]:
def fetch(schema, start, end, tries=4):
    """get_range with backoff. Streaming API - no batch job, nothing in the download centre."""
    for i in range(tries):
        try:
            return client.timeseries.get_range(dataset=DATASET, symbols=[SYMBOL],
                                               stype_in=STYPE, schema=schema,
                                               start=start, end=end)
        except Exception as ex:
            if i == tries - 1:
                raise
            wait = 2 ** i
            print(f"    retry in {wait}s ({type(ex).__name__}: {ex})")
            time.sleep(wait)


def condense_ny(df, day):
    """Long format: ts (ET, minute), side, price, size - one row per level per minute."""
    df = _front_only(df.reset_index())
    tcol = "ts_event" if "ts_event" in df.columns else "ts_recv"
    ts = pd.to_datetime(df[tcol], utc=True, format="mixed")
    df = df.assign(_m=ts.dt.floor("1min"), _t=ts).sort_values("_t")
    last = df.groupby("_m", as_index=False).tail(1)

    frames = []
    for i in range(10):
        for side in ("bid", "ask"):
            pxc, szc = f"{side}_px_{i:02d}", f"{side}_sz_{i:02d}"
            if pxc in last.columns and szc in last.columns:
                sub = last[["_m", pxc, szc]].rename(columns={pxc: "price", szc: "size"})
                sub["side"] = side
                frames.append(sub)
    if not frames:
        return None
    out = pd.concat(frames, ignore_index=True).dropna(subset=["price", "size"])
    out["price"] = np.round(_decode_px(out["price"]), 2)
    out = out[(out["size"] > 0) & (out["price"] > 0)]
    if out.empty:
        return None
    out["size"] = out["size"].astype("int64")
    out = out.rename(columns={"_m": "ts"})
    # ET, matching data/reference/depth_2025|2026 exactly
    out["ts"] = out["ts"].dt.tz_convert(NY)
    return out[["ts", "side", "price", "size"]].sort_values(["ts", "side", "price"]).reset_index(drop=True)


def condense_london(df, day):
    """Wide format: raw Databento columns, one row per minute, ts_event floored to the minute.

    scripts/london_depth.py reads ts_event as the minute label and bid_px_NN/ask_px_NN
    directly, so prices must already be decoded floats here.
    """
    df = _front_only(df.reset_index())
    tcol = "ts_event" if "ts_event" in df.columns else "ts_recv"
    ts = pd.to_datetime(df[tcol], utc=True, format="mixed")
    df = df.assign(_m=ts.dt.floor("1min"), _t=ts).sort_values("_t")
    last = df.groupby("_m", as_index=False).tail(1).copy()
    if last.empty:
        return None
    for c in [c for c in last.columns if "_px_" in c or c == "price"]:
        last[c] = np.round(_decode_px(last[c]), 2)
    last["ts_event"] = last["_m"]                       # minute label, UTC
    last = last.drop(columns=["_m", "_t"]).reset_index(drop=True)

    # Match the committed depth_london/ column order exactly. london_depth.py reads by
    # name so this is cosmetic to it, but keeping the artifacts identical in shape means
    # a new file can be concatenated with an old one without a reindex.
    head = ["ts_event", "ts_recv", "rtype", "publisher_id", "instrument_id", "action",
            "side", "depth", "price", "size", "flags", "ts_in_delta", "sequence"]
    lvls = [f"{s}_{f}_{i:02d}" for i in range(10) for f in ("px", "sz", "ct") for s in ("bid", "ask")]
    order = [c for c in head + lvls + ["symbol"] if c in last.columns]
    return last[order + [c for c in last.columns if c not in order]]


def condense_trades(df, day):
    """Footprint: per (minute, price, aggressor side) volume + tick count.

    side 'B' = buy aggressor, 'A' = sell aggressor (verified empirically in
    scripts/champion_journal_cvd.py: CVD = B - A). volume/trades are cast to int64 -
    the DBN export delivers uint32, which silently wraps on the B-A subtraction.
    """
    df = _front_only(df.reset_index())
    tcol = "ts_event" if "ts_event" in df.columns else "ts_recv"
    df = df[df["side"].isin(["B", "A"])].copy()
    if df.empty:
        return None, 0.0
    df["price"] = np.round(_decode_px(df["price"]), 2)
    df["size"] = pd.to_numeric(df["size"], errors="coerce").fillna(0).astype("int64")

    # Front-month band clean: drop calendar-spread / back-month prints that survive the
    # continuous pull. Volume-weighted median +/- 700pt, the documented fallback in
    # data/reference/cvd/README.md.
    tot = int(df["size"].sum())
    vwm = np.average(df["price"], weights=df["size"]) if tot else 0.0
    keep = df["price"].between(vwm - 700, vwm + 700)
    dropped = 1.0 - (df.loc[keep, "size"].sum() / tot if tot else 1.0)
    df = df[keep]
    if df.empty:
        return None, dropped

    # Nanosecond resolution to match the committed footprint_*.parquet files exactly
    # (pandas 3 defaults new datetimes to microseconds; concat across resolutions is
    # legal but the stored dtype should not silently drift between pulls).
    df["ts_minute"] = (pd.to_datetime(df[tcol], utc=True, format="mixed")
                         .dt.floor("1min").astype("datetime64[ns, UTC]"))
    g = (df.groupby(["ts_minute", "price", "side"], as_index=False)
           .agg(volume=("size", "sum"), trades=("size", "size")))
    g["volume"] = g["volume"].astype("int64")
    g["trades"] = g["trades"].astype("int64")
    return g, dropped

## 5 — Pull and condense

The long cell. Each day is fetched, condensed immediately, and the raw frame freed before the
next request — so ~150 KB/day lands on disk instead of ~1 GB. Days already written are skipped,
so a disconnect costs only the day in flight: just re-run this cell.

Budget roughly 2-4 hours for the full sample. Set `LIMIT_DAYS = 3` in step 2 first if you want
to check the plumbing before committing to the whole run.

In [ ]:
todo = DAYS[:LIMIT_DAYS] if LIMIT_DAYS else DAYS
manifest, fp_by_month, t0 = [], {}, time.time()
print(f"pulling {len(todo)} of {len(DAYS)} sealed days\n")

for n, day in enumerate(todo, 1):
    ny_path  = f"{OUTDIR}/depth_ny/nq_depth_{day}_ny.csv"
    lon_path = f"{OUTDIR}/depth_london/glbx-mdp3-{day.replace('-', '')}.mbp-10_condensed.csv"
    rec = {"day": day, "ny_rows": 0, "london_rows": 0, "fp_rows": 0, "dropped_pct": np.nan}
    el = time.time() - t0
    print(f"[{n:>3}/{len(DAYS)}] {day}  ({el/60:.1f}m elapsed)", flush=True)

    # ---- NY depth ----
    if PULL_NY_DEPTH and not os.path.exists(ny_path):
        try:
            s, e = utc_window(day, *DEPTH_WINDOWS["ny"])
            df = fetch("mbp-10", s, e).to_df()
            out = condense_ny(df, day) if len(df) else None
            del df; gc.collect()
            if out is not None:
                out.to_csv(ny_path, index=False)
                rec["ny_rows"] = len(out)
                print(f"    ny      {len(out):>7,} rows -> {os.path.basename(ny_path)}")
            else:
                print("    ny      no data (holiday?)")
            del out; gc.collect()
        except Exception as ex:
            print(f"    ny      FAILED: {type(ex).__name__}: {ex}")
    elif os.path.exists(ny_path):
        rec["ny_rows"] = -1
        print("    ny      skip (exists)")

    # ---- London depth ----
    if PULL_LONDON_DEPTH and not os.path.exists(lon_path):
        try:
            s, e = utc_window(day, *DEPTH_WINDOWS["london"])
            df = fetch("mbp-10", s, e).to_df()
            out = condense_london(df, day) if len(df) else None
            del df; gc.collect()
            if out is not None:
                out.to_csv(lon_path, index=False)
                rec["london_rows"] = len(out)
                print(f"    london  {len(out):>7,} rows -> {os.path.basename(lon_path)}")
            else:
                print("    london  no data (holiday?)")
            del out; gc.collect()
        except Exception as ex:
            print(f"    london  FAILED: {type(ex).__name__}: {ex}")
    elif os.path.exists(lon_path):
        rec["london_rows"] = -1
        print("    london  skip (exists)")

    # ---- trades / footprint ----
    if PULL_TRADES:
        mo = day[:7]
        mo_path = f"{OUTDIR}/cvd/footprint_holdout_{mo}.parquet"
        done = os.path.exists(mo_path) and mo not in fp_by_month
        if done:
            print("    trades  skip (month file exists)")
        else:
            try:
                s, e = trades_window(day)
                df = fetch("trades", s, e).to_df()
                out, dropped = condense_trades(df, day) if len(df) else (None, 0.0)
                del df; gc.collect()
                if out is not None:
                    fp_by_month.setdefault(mo, []).append(out)
                    rec["fp_rows"] = len(out)
                    rec["dropped_pct"] = round(dropped * 100, 3)
                    print(f"    trades  {len(out):>7,} minute-rows "
                          f"(band-dropped {dropped*100:.2f}%)")
                else:
                    print("    trades  no data")
                del out; gc.collect()
            except Exception as ex:
                print(f"    trades  FAILED: {type(ex).__name__}: {ex}")

    manifest.append(rec)
    time.sleep(0.5)   # be polite to the rate limiter

# ---- flush monthly footprint files ----
for mo, parts in fp_by_month.items():
    p = f"{OUTDIR}/cvd/footprint_holdout_{mo}.parquet"
    allp = pd.concat(parts, ignore_index=True)
    if os.path.exists(p):
        allp = pd.concat([pd.read_parquet(p), allp], ignore_index=True)
    allp = (allp.groupby(["ts_minute", "price", "side"], as_index=False)
                .agg(volume=("volume", "sum"), trades=("trades", "sum")))
    allp["volume"] = allp["volume"].astype("int64")
    allp["trades"] = allp["trades"].astype("int64")
    allp.to_parquet(p, index=False)
    print(f"wrote {os.path.basename(p)}: {len(allp):,} rows")

pd.DataFrame(manifest).to_csv(f"{OUTDIR}/MANIFEST.csv", index=False)
print(f"\ndone in {(time.time()-t0)/60:.1f} min")

## 6 — Verify, zip, download

Coverage check first — a silently missing day is worse than a loud failure — then zip and
download. Send the zip back to Claude.

In [ ]:
import glob

ny_have  = {os.path.basename(f).split("_")[2] for f in glob.glob(f"{OUTDIR}/depth_ny/nq_depth_*_ny.csv")}
lon_have = {os.path.basename(f).split(".")[0].split("-")[-1] for f in glob.glob(f"{OUTDIR}/depth_london/*.csv")}
lon_have = {f"{d[:4]}-{d[4:6]}-{d[6:]}" for d in lon_have if len(d) == 8}

want = set(DAYS)
print(f"sealed days      : {len(want)}")
print(f"ny depth files   : {len(ny_have)}   missing {len(want - ny_have)}")
print(f"london files     : {len(lon_have)}   missing {len(want - lon_have)}")
for label, have in (("ny", ny_have), ("london", lon_have)):
    miss = sorted(want - have)
    if miss:
        print(f"  {label} missing: {miss[:12]}{' ...' if len(miss) > 12 else ''}")

fps = sorted(glob.glob(f"{OUTDIR}/cvd/*.parquet"))
print(f"\nfootprint files  : {len(fps)}")
for f in fps:
    x = pd.read_parquet(f)
    ts = pd.to_datetime(x.ts_minute, utc=True)
    print(f"  {os.path.basename(f):<34}{len(x):>9,} rows  "
          f"{ts.min():%Y-%m-%d} -> {ts.max():%Y-%m-%d}  "
          f"vol={int(x.volume.sum()):,}  dtypes={x.volume.dtype}/{x.trades.dtype}")

size_mb = sum(os.path.getsize(f) for f in glob.glob(f"{OUTDIR}/**/*", recursive=True)
              if os.path.isfile(f)) / 1e6
print(f"\ntotal condensed on disk: {size_mb:.1f} MB")

with zipfile.ZipFile(ZIPOUT, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUTDIR}/**/*", recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, OUTDIR))

print(f"{ZIPOUT}  ->  {os.path.getsize(ZIPOUT)/1e6:.1f} MB")
print("\nDownload it from the Colab file browser on the left, then hand it back to Claude.")
print("Unpacks into: depth_ny/  depth_london/  cvd/  MANIFEST.csv")

try:
    from google.colab import files
    files.download(ZIPOUT)
except Exception as ex:
    print(f"(auto-download unavailable: {ex} - grab it from the file browser)")

## What happens to it back in the repo

```
depth_ny/*.csv       -> data/reference/depth_2023_24/
depth_london/*.csv   -> data/reference/depth_london/
cvd/*.parquet        -> data/reference/cvd/
```

Then the canon is scored **unchanged** — same frozen 2025 thresholds, no refits, no new knobs.
Any threshold that moves turns this from a holdout into a fit.

Raw data never enters the repo; only these condensed artifacts do.